# Cleanup

Removes everything created by the pattern notebooks, most expensive first:

1. **Endpoints** (billed per instance-hour), endpoint configs, and models
2. **Model Packages and Model Package Groups** created by the auto-sync
3. **MLflow registered models**
4. (optional) the **MLflow app** itself
5. (optional) the **ECR repository** from pattern 1
6. (optional) **S3 artifacts** — training job outputs, MLflow artifact store, repacked archives
7. (optional) the **CloudFormation stack**, if you used it to provision the environment

Remember to also **stop or delete the JupyterLab space** when finished.

If you ran all three pattern notebooks without their teardown cells, the
**Deployments → Endpoints** view in Studio looks like this — three `ml.m5.xlarge`
endpoints billing per instance-hour:

![Three in-service endpoints in SageMaker Studio, one per deployment pattern](img/deployed-endpoints.png)

In [ ]:
%store -r model_base_name
%store -r mlflow_app_arn
%store -r mlflow_app_name
%store -r region

import boto3
import mlflow

sm_client = boto3.client("sagemaker")
mlflow.set_tracking_uri(mlflow_app_arn)
mlflow_client = mlflow.MlflowClient()

PREFIX = model_base_name  # all resources in this repo share this prefix
print(f"Cleaning up resources with prefix: {PREFIX}")

## Step 1: Endpoints, endpoint configs, models

In [ ]:
for ep in sm_client.list_endpoints(NameContains=PREFIX)["Endpoints"]:
    print(f"Deleting endpoint {ep['EndpointName']}")
    sm_client.delete_endpoint(EndpointName=ep["EndpointName"])

for cfg in sm_client.list_endpoint_configs(NameContains=PREFIX)["EndpointConfigs"]:
    print(f"Deleting endpoint config {cfg['EndpointConfigName']}")
    sm_client.delete_endpoint_config(EndpointConfigName=cfg["EndpointConfigName"])

for m in sm_client.list_models(NameContains=PREFIX)["Models"]:
    print(f"Deleting model {m['ModelName']}")
    sm_client.delete_model(ModelName=m["ModelName"])

print("Done.")

## Step 2: Model Packages and Model Package Groups (patterns 2 and 3)

The auto-sync appends a short hash suffix to group names, so we match on
`NameContains`.

In [ ]:
import time

groups = sm_client.list_model_package_groups(NameContains=PREFIX)[
    "ModelPackageGroupSummaryList"
]
for g in groups:
    name = g["ModelPackageGroupName"]
    for pkg in sm_client.list_model_packages(ModelPackageGroupName=name)[
        "ModelPackageSummaryList"
    ]:
        print(f"Deleting model package {pkg['ModelPackageArn']}")
        sm_client.delete_model_package(ModelPackageName=pkg["ModelPackageArn"])

time.sleep(10)  # let the deletes propagate
for g in groups:
    name = g["ModelPackageGroupName"]
    print(f"Deleting model package group {name}")
    sm_client.delete_model_package_group(ModelPackageGroupName=name)

print("Done.")

## Step 3: MLflow registered models

In [ ]:
for rm in mlflow_client.search_registered_models(f"name LIKE '{PREFIX}%'"):
    print(f"Deleting registered model {rm.name}")
    mlflow_client.delete_registered_model(rm.name)

print("Done.")

## Step 4 (optional): Delete the MLflow app

Uncomment if you created the app just for this walkthrough.

In [ ]:
# sm_client.delete_mlflow_app(Arn=mlflow_app_arn)
# print(f"Deleting MLflow app {mlflow_app_name}")

## Step 5 (optional): ECR repository from pattern 1

`mlflow sagemaker build-and-push-container` created an `mlflow-pyfunc` repository.

In [ ]:
# ecr = boto3.client("ecr")
# ecr.delete_repository(repositoryName="mlflow-pyfunc", force=True)
# print("Deleted ECR repository mlflow-pyfunc")

## Step 6 (optional): S3 artifacts

The walkthrough also leaves artifacts in S3 that continue to accrue storage costs:

- the **training job outputs** from `00_setup_and_train.ipynb`,
- the **MLflow artifact store** contents (only if you also deleted the MLflow app above),
- the **repacked model archive** created by `ModelBuilder` in pattern 2.

Uncomment below to list them first, then delete what you no longer need.


In [ ]:
# import sagemaker
# bucket = sagemaker.Session().default_bucket()
# s3 = boto3.resource("s3").Bucket(bucket)
#
# # List everything the walkthrough wrote under the shared prefix
# for obj in s3.objects.filter(Prefix=PREFIX):
#     print(obj.key)
#
# # Delete after reviewing the list above
# # s3.objects.filter(Prefix=PREFIX).delete()
# # print(f"Deleted s3://{bucket}/{PREFIX}*")


## Step 7 (optional): CloudFormation stack

If you provisioned the environment with the CloudFormation template in `cfn/`,
delete the stack when you are completely done — it removes the SageMaker domain
resources it created. **Note:** stop or delete the JupyterLab space you are running
this notebook from *before* deleting the stack.


In [ ]:
# cfn = boto3.client("cloudformation")
# cfn.delete_stack(StackName="deploy-mlflow-models")
# print("Deleting CloudFormation stack deploy-mlflow-models")
